# Creating a Spotify Playlist

#### We will use Spotify

In [2]:
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
import os
import requests
from bs4 import BeautifulSoup

In [3]:
env_path = str(Path.cwd().parent)+'/.env'
load_dotenv(env_path)

True

In [4]:
URL = 'https://appbrewery.github.io/bakeboard-hot-100/'
USER_AGENT = os.getenv("USER-AGENT")

In [5]:
#So for check the date we will use a function
def check_date():
    #we keep asking until the user digit a valid date
    while True:
        #ask the year
        year = input('Which year do you want to travel to? Type the data in this format YYYY-MM-DD (example 2020-04-18): ')
        #and try-except
        try:
        #we use the library and method strptime, to strip our date into a valid one
            date = datetime.strptime(year, '%Y-%m-%d')
            print('Valid date!')
            #if we do not have an error we return the date in the valid format
            return date
        #if we have an error we return that value
        except ValueError:
            print('Invalid date. Please use YYYY-MM-DD.')

In [6]:
def scrape(date):
    #so the original url is
    # 'https://appbrewery.github.io/bakeboard-hot-100/2026-04-18/'
    #the last part is the date, (is the one we asked the user)
    #so we just need to add it
    #we need to add a header, (i really do not know why but we will add it)
    header = {"User-Agent": USER_AGENT}
    response = requests.get(url=URL+date+'/', headers=header)
    response.raise_for_status()
    return response.text

In [7]:
def get_songs(data):
    #we need to parser the data with beautiful soup
    soup = BeautifulSoup(data,'html.parser')
    #for now we will get the name of the songs, 
    names = [a.get_text() for a in soup.find_all(name='h3', class_='chart-entry__title')]

    #but if we need it we can get the autor and ranking
    authors = [a.get_text() for a in soup.find_all(name='span', class_='chart-entry__artist')]
    #this as a number but we actually do not need this since are order
    #rankings = [int(a.get_text()) for a in soup.find_all(name='span', class_='chart-entry__rank-number')]
    #we can put it all together in a list of dictionaries, add(rankings) if we like to
    songs = [{'title':title, 'author':author} for title, author in zip(names, authors)]
    #both are list but one have all the three data at once
    return songs

In [8]:
def main():
    #first we need to request the date to the user
    correct_date = check_date()
    correct_date = correct_date.date()
    #for now if the date is correct we print it
    print(correct_date)
    #next we will scrape
    data_web = scrape(str(correct_date))
    #print the songs names
    list_songs = get_songs(data_web)
    print(list_songs)

In [21]:
main()

Valid date!
2020-08-15
[{'title': 'Watermelon Sugar', 'author': 'Harry Styles'}, {'title': 'Rockstar', 'author': 'DaBaby Featuring Roddy Ricch'}, {'title': 'Whats Poppin', 'author': 'Jack Harlow Featuring DaBaby, Tory Lanez & Lil Wayne'}, {'title': 'Blinding Lights', 'author': 'The Weeknd'}, {'title': 'Roses', 'author': 'SAINt JHN'}, {'title': 'My Future', 'author': 'Billie Eilish'}, {'title': 'Savage Love (Laxed - Siren Beat)', 'author': 'Jawsh 685 x Jason Derulo'}, {'title': 'Cardigan', 'author': 'Taylor Swift'}, {'title': 'Go Crazy', 'author': 'Chris Brown & Young Thug'}, {'title': 'Blueberry Faygo', 'author': 'Lil Mosey'}, {'title': 'Before You Go', 'author': 'Lewis Capaldi'}, {'title': 'Adore You', 'author': 'Harry Styles'}, {'title': 'Savage', 'author': 'Megan Thee Stallion'}, {'title': 'Break My Heart', 'author': 'Dua Lipa'}, {'title': 'I Hope', 'author': 'Gabby Barrett Featuring Charlie Puth'}, {'title': 'Circles', 'author': 'Post Malone'}, {'title': 'Come & Go', 'author': 'Jui

### So we have the name of the songs, the artist and a correct date that we validate, we have two ways 
* Youtube Music
* Spotify

### We will do for now Spotify way

## **Spotify**

## Create an APP & Authenticate 
# STEP ONE
It is important to mentiont that this can work, with the creation of an app, but unfortunally this is only for premium users, so be aware of that

And after creating our app we will see our client ID, and client secret

After reading the documentation for use the API our authorization will need to be
> Authorization Code with PKCE Flow

This is for mobile app, single page web apps, or any other type of application where the client secret can’t be safely stored. (we do not have an own server)

But it is quite complicated, so instead we're going to use one of the most popular Python Spotify modules
>Spotipy

## Create an Spotify API client

So after reading the documentarion for this module, we will use this class

**class spotipy.oauth2.SpotifyOAuth**

That implements Authorization Code Flow for Spotify’s OAuth implementation.

**How do we create a SpotifyOAuth object?**

* https://spotipy.readthedocs.io/en/2.26.0/#spotipy.oauth2.SpotifyOAuth

There is an important step, **the URL**, as we know the instructions says that we use **example.com**, but we can use **"http://127.0.0.1:8888/callback"** is this possible without a server? Well yes:

* **Spotipy** starts a temporary local server on port 8888.
* Spotify authorization page
  * Your browser opens and asks you to authorize the application.
* Spotify redirects to 127.0.0.1:8888/callback
  * The temporary server receives the authorization code.
* Spotipy receives the token
  * The token is saved in your token.txt file.

This will work for every time leets see why

* **Situation**	|   **What** **happens**
* First execution	|   Browser authorization is required.
* Second execution	|   Spotipy reads the cached token.
* Third execution	|   Same process; no new authorization needed.
* Access token expires	|   Spotipy can refresh it automatically.
* Token cache is deleted	|   Authorization is required again.

Spotipy checks the cache and refreshes expired tokens when possible.

Also for the app in the Spotify Developer Dashboard we need to do some things

* The redirect url need to be the same http://127.0.0.1:8888/callback
* We need to add an user (we need to add ourselfs (email/username)) 

In [20]:
env_path = str(Path.cwd().parent)+'/.env'
CACHE_PATH = Path.cwd() / "token.txt"
load_dotenv(env_path)
ID_C = os.getenv("CLIENT_ID")
SE_C = os.getenv("CLIENT_SEC")
URL = 'http://127.0.0.1:8888/callback'

In [12]:
import spotipy 
from spotipy.oauth2 import SpotifyOAuth, CacheFileHandler

In [ ]:
sp = spotipy.Spotify(
    auth_manager=SpotifyOAuth(
        client_id=ID_C,
        client_secret=SE_C,
        redirect_uri=URL,
        #first scope gives our application permission to create and modify private playlists.
        #second allows our application to retrieve the authenticated user's profile.
        scope="playlist-modify-private user-read-private",
        cache_handler=CacheFileHandler(cache_path=str(CACHE_PATH)),
        show_dialog=True
    )
)
# Get the authenticated Spotify user
user = sp.current_user()

Opening in existing browser session.
[28 zypak-sandbox] Failed to wait for supervisor exit reply: Connection reset by peer (errno 104)


[26:26:0100/000000.238306:ERROR:content/zygote/zygote_linux.cc:662] write: Broken pipe (32)


In [29]:
#Optional
user_id = user["id"]
#print(user["display_name"])
#print(user_id)

### And we got it, the first time I gave the authorization, but for the next times, it will read the token.txt file now we can move on to the next part, we can see what the output for the first time or if we delete the file is (we will refactor later)

## Search for the Songs from Step 1

In the documentation it says that for 

**Creates a playlist for the current user** we use:

current_user_playlist_create(name, public=True, collaborative=False, description='')
https://spotipy.readthedocs.io/en/2.26.0/#spotipy.client.Spotify.current_user_playlist_create


In [ ]:
new_playlist = sp.current_user_playlist_create(
    #we can pass the year also
    name= f'Billboard hot 100 of {...}',
    public=False
    #the rest we do not need it
)

In [ ]:
# To search a song we have the method call search()

results = sp.search(
    #search query
    #list_songs, is the one we created when we scrape the page 
    q=f'track:{list_songs["title"]} artist:{list_songs["author"]}',
    #specifically for songs (nbot albums/podcasts else)
    type="track",
    #(just one result)
    limit=1
)

#this will return a dictionary like this
results["tracks"]["items"]
#inside items we will find the matching tracks
results["tracks"]["items"][0]["uri"]
#the 'uri' IDENTIFIES the spotify track --> is use to add it to our playlist

In [ ]:
#So that is just for one song to make it for all the songs
track_uris = []

for song in list_songs:
    results = sp.search(
        q=f'track:{song["title"]} artist:{song["author"]}',
        type="track",
        limit=1
    )

    tracks = results["tracks"]["items"]

    if tracks:
        track_uris.append(tracks[0]["uri"])
    else:
        print(f'Song not found: {song["title"]}')

    #A song not being found doesn't necessarily raise an exception. Spotify can simply return an empty list.

In [ ]:
#after search the songs we can add it to the playlist
sp.playlist_add_items(
    #this will identify the playlist we just created
    playlist_id=new_playlist["id"],
    #and this will add all the songs we search before
    items=track_uris
)

# We are going to do it all

In [18]:
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
import os
import requests
from bs4 import BeautifulSoup
import spotipy 
from spotipy.oauth2 import SpotifyOAuth, CacheFileHandler

In [19]:
env_path = str(Path.cwd().parent)+'/.env'
load_dotenv(env_path)

ID_C = os.getenv("CLIENT_ID")
SE_C = os.getenv("CLIENT_SEC")
USER_AGENT = os.getenv("USER-AGENT")
CACHE_PATH = Path.cwd() / "token.txt"
URL_BILLBOARD= 'https://appbrewery.github.io/bakeboard-hot-100/'
URL_REPI = 'http://127.0.0.1:8888/callback'

In [20]:
#Scheck valid date
def check_date():
    while True:
        input_date = input('Which year do you want to travel to? Type the data in this format YYYY-MM-DD (example 2020-04-18): ')
        try:
            date = datetime.strptime(input_date, '%Y-%m-%d')
            print('Valid date!')
            return date
        except ValueError:
            print('Invalid date. Please use YYYY-MM-DD.')

In [21]:
def scrape(date):
    header = {"User-Agent": USER_AGENT}
    response = requests.get(url=URL_BILLBOARD+date+'/', headers=header)
    response.raise_for_status()
    return response.text

In [22]:
def get_songs(data):
    soup = BeautifulSoup(data,'html.parser')
    names = [a.get_text() for a in soup.find_all(name='h3', class_='chart-entry__title')]
    authors = [a.get_text() for a in soup.find_all(name='span', class_='chart-entry__artist')]
    songs = [{'title':title, 'author':author} for title, author in zip(names, authors)]
    return songs

In [23]:
#create user for spotify
def create_user():
    sp = spotipy.Spotify(
        auth_manager=SpotifyOAuth(
            client_id=ID_C,
            client_secret=SE_C,
            redirect_uri=URL_REPI,
            #first scope gives our application permission to create and modify private playlists.
            #second allows our application to retrieve the authenticated user's profile.
            scope="playlist-modify-private user-read-private",
            cache_handler=CacheFileHandler(cache_path=str(CACHE_PATH)),
            show_dialog=True
        )
    )
    return sp

In [ ]:
def search_songs(list_songs, sp):
    track_uris = []
    for song in list_songs:
        results = sp.search(
            q=f'track:{song["title"]} artist:{song["author"]}',
            type="track",
            limit=1
        )

        tracks = results["tracks"]["items"]

        if tracks:
            track_uris.append(tracks[0]["uri"])
        else:
            print(f'Song not found: {song["title"]}')

    #after all the loop we return the list of tracks_uris
    return track_uris

In [29]:
def main():
    #get the date from user
    correct_date = check_date()
    print(correct_date.date())

    #scrape from billboard
    data_web = scrape(str(correct_date.date()))

    #get the dictionary (song name, author)
    list_songs = get_songs(data_web)
    print(list_songs)

    # Create the Spotify API client
    sp = create_user()
    # Get the authenticated user's information
    sp.current_user()

    #after this then we are going to make a search for the songs in spotify
    songs = search_songs(list_songs, sp)
    print(songs)

    #then we create the playlist
    new_playlist = sp.current_user_playlist_create(
                        #We can pass the year with this
                        name= f'Billboard hot 100 of {str(correct_date).split('-')[0]}',
                        public=False
                    )   

    #add the songs to the new list
    sp.playlist_add_items(
        playlist_id=new_playlist["id"],
        items=songs
    )

    #All done check it
    print('Playlist created, and songs added it, check now on your profile')

In [30]:
main()

Valid date!
2020-08-15
[{'title': 'Watermelon Sugar', 'author': 'Harry Styles'}, {'title': 'Rockstar', 'author': 'DaBaby Featuring Roddy Ricch'}, {'title': 'Whats Poppin', 'author': 'Jack Harlow Featuring DaBaby, Tory Lanez & Lil Wayne'}, {'title': 'Blinding Lights', 'author': 'The Weeknd'}, {'title': 'Roses', 'author': 'SAINt JHN'}, {'title': 'My Future', 'author': 'Billie Eilish'}, {'title': 'Savage Love (Laxed - Siren Beat)', 'author': 'Jawsh 685 x Jason Derulo'}, {'title': 'Cardigan', 'author': 'Taylor Swift'}, {'title': 'Go Crazy', 'author': 'Chris Brown & Young Thug'}, {'title': 'Blueberry Faygo', 'author': 'Lil Mosey'}, {'title': 'Before You Go', 'author': 'Lewis Capaldi'}, {'title': 'Adore You', 'author': 'Harry Styles'}, {'title': 'Savage', 'author': 'Megan Thee Stallion'}, {'title': 'Break My Heart', 'author': 'Dua Lipa'}, {'title': 'I Hope', 'author': 'Gabby Barrett Featuring Charlie Puth'}, {'title': 'Circles', 'author': 'Post Malone'}, {'title': 'Come & Go', 'author': 'Jui

# Ok thanks great it works, but the problem is that for some songs that already are on spotify did not match the result, and thats because the name of the artist is like
DaBaby **Featuring** Roddy Ricch',

Jack Harlow **Featuring** DaBaby, Tory Lanez & Lil Wayne'

...

SO we can fix that with the next implementation

In [ ]:
import re

In [ ]:
def search_songs(list_songs, sp):
    track_uris = []

    for song in list_songs:

        # Extract the main artist
        #re.split uses regular expressions to determine where to split
        main_artist = re.split(
            #this is the expression
            r'\b(?:featuring|feat\.?|ft\.?)\b',
            #in the string (current author)
            #"DaBaby Featuring Roddy Ricch"
            # [DaBaby, Roddy Ricch]
            song["author"],
            #if the string has multiple featured artist, we will only separete the main artist from the rest
            maxsplit=1,
            flags=re.IGNORECASE
        )[0].strip()

        # First attempt: title + main artist
        results = sp.search(
            q=f'track:{song["title"]} artist:{main_artist}',
            type="track",
            limit=5
        )

        tracks = results["tracks"]["items"]

        # Second attempt if we have no URI 
        if not tracks:
            #we do a search only with title
            results = sp.search(
                q=f'track:{song["title"]}',
                type="track",
                #for 10 songs
                limit=10
            )
            tracks = results["tracks"]["items"]

        # Check whether the main artist is present
        #if finds a matching track it returns the track if not then None
        #see it like
        #Give me the first matching track. If you cannot find one, give me None.
        match = next(
            (
                #(for track in tracks) iterateas every track returned by spotify
                # track () produces tracks one at time, only when it staisfy the condition
                track for track in tracks
                if main_artist.casefold() in [
                    #A list comprehension wheres creates a list of artist names
                    #.casefold() is to compare strings without worryinh about uppercase or lowercase differences
                    artist["name"].casefold()
                    for artist in track["artists"]
                ]
            ),
            None
        )

        if match:
            track_uris.append(match["uri"])
        else:
            print(f'Could not confidently match: {song["title"]} - {song["author"]}')

    return track_uris

# And also if the playlist already exist on the same year, how we can avoid to create another one with same name, instead just updateing the songs, how?

In [ ]:
def find_playlist(sp, playlist_name, user_id):
    #Spotify returns a maximum of 50 playlists per request.
    playlists = sp.current_user_playlists(limit=50)

    while playlists:
        for playlist in playlists["items"]:

            if (playlist["name"] == playlist_name
                    and playlist["owner"]["id"] == user_id):

                return playlist

        # Get the next page of playlists (use of pagination)
        playlists = sp.next(playlists)
        #We continue until there are no more pages.

    return None

## But in our creation of the API of spotify we need to add **playlist-read-private** as one additional permission

In [31]:
scope="playlist-modify-private playlist-read-private user-read-private" # like this

# Lets Upgrade everything

In [32]:
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
import os
import requests
from bs4 import BeautifulSoup
import spotipy 
from spotipy.oauth2 import SpotifyOAuth, CacheFileHandler
import re

In [33]:
env_path = str(Path.cwd().parent)+'/.env'
load_dotenv(env_path)

ID_C = os.getenv("CLIENT_ID")
SE_C = os.getenv("CLIENT_SEC")
USER_AGENT = os.getenv("USER-AGENT")
CACHE_PATH = Path.cwd() / "token.txt"
URL_BILLBOARD= 'https://appbrewery.github.io/bakeboard-hot-100/'
URL_REPI = 'http://127.0.0.1:8888/callback'

In [34]:
#Scheck valid date
def check_date():
    while True:
        input_date = input('Which year do you want to travel to? Type the data in this format YYYY-MM-DD (example 2020-04-18): ')
        try:
            date = datetime.strptime(input_date, '%Y-%m-%d')
            print('Valid date!')
            return date
        except ValueError:
            print('Invalid date. Please use YYYY-MM-DD.')

In [35]:
def scrape(date):
    header = {"User-Agent": USER_AGENT}
    response = requests.get(url=URL_BILLBOARD+date+'/', headers=header)
    response.raise_for_status()
    return response.text

In [36]:
def get_songs(data):
    soup = BeautifulSoup(data,'html.parser')
    names = [a.get_text() for a in soup.find_all(name='h3', class_='chart-entry__title')]
    authors = [a.get_text() for a in soup.find_all(name='span', class_='chart-entry__artist')]
    songs = [{'title':title, 'author':author} for title, author in zip(names, authors)]
    return songs

In [37]:
#create user for spotify
def create_user():
    sp = spotipy.Spotify(
        auth_manager=SpotifyOAuth(
            client_id=ID_C,
            client_secret=SE_C,
            redirect_uri=URL_REPI,
            #first scope gives our application permission to create and modify private playlists.
            #second allows our application to retrieve the authenticated user's profile.
            scope="playlist-modify-private playlist-read-private user-read-private",
            cache_handler=CacheFileHandler(cache_path=str(CACHE_PATH)),
            show_dialog=True
        )
    )
    return sp

In [38]:
def search_songs(list_songs, sp):
    track_uris = []

    for song in list_songs:

        main_artist = re.split(
            r'\b(?:featuring|feat\.?|ft\.?)\b',
            song["author"],
            maxsplit=1,
            flags=re.IGNORECASE
        )[0].strip()

        # First attempt
        results = sp.search(
            q=f'track:{song["title"]} artist:{main_artist}',
            type="track",
            limit=5
        )

        tracks = results["tracks"]["items"]

        # Second attempt 
        if not tracks:
            results = sp.search(
                q=f'track:{song["title"]}',
                type="track",
                limit=10
            )
            tracks = results["tracks"]["items"]

        match = next(
            (
                track for track in tracks
                if main_artist.casefold() in [
                    artist["name"].casefold()
                    for artist in track["artists"]
                ]
            ),
            None
        )

        if match:
            track_uris.append(match["uri"])
        else:
            print(f'Could not confidently match: {song["title"]} - {song["author"]}')
    return track_uris

In [39]:
def find_playlist(sp, playlist_name, user_id):
    playlists = sp.current_user_playlists(limit=50)

    while playlists:
        for playlist in playlists["items"]:
            if (playlist["name"] == playlist_name
                    and playlist["owner"]["id"] == user_id):
                return playlist
        playlists = sp.next(playlists)
    return None

In [40]:
def main():
    #get the date from user
    correct_date = check_date()
    print(correct_date.date())

    #scrape from billboard
    data_web = scrape(str(correct_date.date()))

    #get the dictionary (song name, author)
    list_songs = get_songs(data_web)
    print(list_songs)

    # Create the Spotify API client
    sp = create_user()
    # Get the authenticated user's information
    user = sp.current_user()
    user_id = user['id']

    #after this then we are going to make a search for the songs in spotify
    songs_uries = search_songs(list_songs, sp)
    print(songs_uries)

    # Prepare playlist name
    playlist_name = f'Billboard Hot 100 - {correct_date.strftime("%Y-%m-%d")}'

    #Check the playlist if exists
    existing_playlist = find_playlist(sp, playlist_name, user_id)
    if existing_playlist:
        print("Playlist already exists!\n We will update the songs")
        playlist_id = existing_playlist['id']
    else:
        print("Creating a new playlist!")
        new_playlist = sp.current_user_playlist_create(
                            #We can pass the year with this
                            name= f'Billboard hot 100 of {str(correct_date).split('-')[0]}',
                            public=False
        )  
        playlist_id = new_playlist['id']

    
    if songs_uries:
        sp.playlist_replace_items(
            playlist_id=playlist_id,
            items=songs_uries
        )
        print(f'Playlist updated successfully! {len(songs_uries)} songs added.')
    else:
        print("No songs were found. Playlist was not modified.")

    #All done check it
    print('Playlist created, and songs added it, check now on your profile')

In [41]:
main()

Valid date!
2020-08-15
[{'title': 'Watermelon Sugar', 'author': 'Harry Styles'}, {'title': 'Rockstar', 'author': 'DaBaby Featuring Roddy Ricch'}, {'title': 'Whats Poppin', 'author': 'Jack Harlow Featuring DaBaby, Tory Lanez & Lil Wayne'}, {'title': 'Blinding Lights', 'author': 'The Weeknd'}, {'title': 'Roses', 'author': 'SAINt JHN'}, {'title': 'My Future', 'author': 'Billie Eilish'}, {'title': 'Savage Love (Laxed - Siren Beat)', 'author': 'Jawsh 685 x Jason Derulo'}, {'title': 'Cardigan', 'author': 'Taylor Swift'}, {'title': 'Go Crazy', 'author': 'Chris Brown & Young Thug'}, {'title': 'Blueberry Faygo', 'author': 'Lil Mosey'}, {'title': 'Before You Go', 'author': 'Lewis Capaldi'}, {'title': 'Adore You', 'author': 'Harry Styles'}, {'title': 'Savage', 'author': 'Megan Thee Stallion'}, {'title': 'Break My Heart', 'author': 'Dua Lipa'}, {'title': 'I Hope', 'author': 'Gabby Barrett Featuring Charlie Puth'}, {'title': 'Circles', 'author': 'Post Malone'}, {'title': 'Come & Go', 'author': 'Jui

[26:26:0100/000000.991979:ERROR:content/zygote/zygote_linux.cc:662] write: Broken pipe (32)


Could not confidently match: Savage Love (Laxed - Siren Beat) - Jawsh 685 x Jason Derulo
Could not confidently match: Go Crazy - Chris Brown & Young Thug
Could not confidently match: Come & Go - Juice WRLD x Marshmello
Could not confidently match: We Paid - Lil Baby & 42 Dugg
Could not confidently match: Rain On Me - Lady Gaga & Ariana Grande
Could not confidently match: Be Like That - Kane Brown With Swae Lee & Khalid
Could not confidently match: Un Dia (One Day) - J Balvin, Dua Lipa, Bad Bunny & Tainy
Could not confidently match: Hate The Other Side - Juice WRLD & Marshmello Featuring Polo G & The Kid LAROI
Could not confidently match: Mamacita - Black Eyed Peas, Ozuna + J.Rey Soul
Could not confidently match: Stuck With U - Ariana Grande & Justin Bieber
Could not confidently match: Life's A Mess - Juice WRLD X Halsey
Could not confidently match: Be Kind - Marshmello & Halsey
Could not confidently match: No Dribble - DaBaby x Stunna 4 Vegas
Could not confidently match: Breaking Me - 

# And with that we improve our matches from 65 to 86

# In the future will be another file will be the Youtube part